# Sentiment Analysis Pipeline — Handling Multiple Aspects

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.special import softmax
import urllib.request
import numpy as np
import pandas as pd
import time
import csv
import re

In [2]:
import warnings
warnings.filterwarnings('ignore')

## 2. Improving Handling of Multiple Aspects

In the previous approach, each comment was analyzed as a whole,  
and aspects were not clearly separated.
In this notebook, we focus on improving how the system handles  
multiple aspects within the same sentence.
The goal is to better associate detected aspects with sentiment predictions.

In [3]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = "@user" if t.startswith("@") and len(t) > 1 else t
        t = "http" if t.startswith("http") else t
        new_text.append(t)
    return " ".join(new_text)

In [4]:
task = "sentiment"
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

labels = []
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"

Loading weights: 100%|█████████████████| 201/201 [00:00<00:00, 16154.50it/s]


In [5]:
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode("utf-8").split("\n")
    csvreader = csv.reader(html, delimiter="\t")
    labels = [row[1] for row in csvreader if len(row) > 1]

In [7]:
def predict(text):
    text = preprocess(text)
    encoded_input = tokenizer(text, return_tensors="pt")
    output = model(**encoded_input)
    scores = output.logits[0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)[::-1]
    # Extract the top score and label
    label = labels[ranking[0]]
    score = scores[ranking[0]]
    return label, float(f"{score:.2f}")

## 3. Designing the Aspect Detection Logic

We implement a function that detects predefined aspects  
inside each comment.

Examples of aspects:
- price
- design
- quality
- service

This is done using simple keyword matching.

This approach is lightweight and allows us to quickly identify  
which parts of the comment relate to specific aspects.

In [16]:
def split_into_phrases(text):
    parts = re.split(r"\s*(?:\.|\bbut\b|\bhowever\b)\s*",text,flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]

In [9]:
ASPECTS = ["design", "performance", "price", "service", "quality", "delivery", "staff"]

In [10]:
def get_aspects(phrase):
    found_aspects = []
    for aspect in ASPECTS:
        if aspect.lower() in phrase.lower():
            found_aspects.append(aspect.lower())
    return found_aspects

In [19]:
def analyze_comment(text):
    results_with_aspects = []
    results_without_aspects = []
    phrases = split_into_phrases(text)
    for phrase in phrases:
        aspects = get_aspects(phrase)
        label, score = predict(phrase)
        if aspects:
            results_with_aspects.append({"Phrase": phrase,"Aspects": aspects,"Label": label,"Score": score})
        else:
            results_without_aspects.append({"Phrase": phrase,"Label": label,"Score": score})
    return {
        "with_aspects": results_with_aspects,
        "without_aspects": results_without_aspects
    }

## 4. Basic Testing

We start by testing the aspect detection logic on simple comments.

This allows us to verify:
- Aspects are correctly detected
- The pipeline behaves as expected

In [21]:
text_2 = "Both the design and the quality are good"
analyze_comment(text_2)

{'with_aspects': [{'Phrase': 'Both the design and the quality are good',
   'Aspects': ['design', 'quality'],
   'Label': 'positive',
   'Score': 0.96}],
 'without_aspects': []}

In [20]:
text = "The design is beautiful and the performance is terrible but I regret buying it. However the delivery was fast."

analyze_comment(text)

{'with_aspects': [{'Phrase': 'The design is beautiful and the performance is terrible',
   'Aspects': ['design', 'performance'],
   'Label': 'negative',
   'Score': 0.48},
  {'Phrase': 'the delivery was fast',
   'Aspects': ['delivery'],
   'Label': 'positive',
   'Score': 0.85}],
 'without_aspects': [{'Phrase': 'I regret buying it',
   'Label': 'negative',
   'Score': 0.86}]}

## 5. Testing on Multiple Data

We apply the pipeline on multiple comments.

For each comment:
- Sentiment is predicted
- Aspects are extracted
- Results are structured into outputs

This helps evaluate how the system behaves on more realistic inputs.

In [22]:
test_comments = [
    # Straightforward + known aspect
    "The design is beautiful.",
    "The price is too expensive.",

    # Straightforward + no known aspect
    "I really loved it.",
    "This was a complete waste of time.",

    # Positive then negative, both with known aspects
    "The design is beautiful but the performance is terrible.",
    "The staff were friendly but the service was very slow.",

    # Negative then positive, both with known aspects
    "The price is high but the quality is excellent.",
    "The performance was bad at first but the service was great.",

    # Complex: one part has aspect, one part has no aspect
    "The design is amazing but I still regret buying it.",
    "I hated the experience at first but the staff were very kind.",

    # Multiple aspects in one sentence part
    "The design and performance are both excellent.",
    "The price and delivery were disappointing.",

    # No aspect at all, mixed sentiment
    "I liked it at first but it became disappointing later.",
    "At the beginning it was confusing but in the end it was useful.",

    # Edge cases
    "The product is okay.",
    "Not bad, but not amazing either.",
    "The service was not terrible, but it was not great.",
    "The delivery was fast however the package looked damaged.",
]

In [29]:
r_aspects = []
r_w_aspects = []

start_time = time.perf_counter()
for comment in test_comments:
    results = analyze_comment(comment)
    if results["with_aspects"]:
        r_aspects.extend(results["with_aspects"])
    if results["without_aspects"]: 
        r_w_aspects.extend(results["without_aspects"])
end_time = time.perf_counter() 
print(f" This function took {end_time - start_time} s to be executed")

 This function took 61.40698782494292 s to be executed


In [33]:
df_aspects = pd.DataFrame(r_aspects)
df_aspects.head(10)

,Phrase,Aspects,Label,Score
0,The design is beautiful,[design],positive,0.98
1,The price is too expensive,[price],negative,0.87
2,The design is beautiful,[design],positive,0.98
3,the performance is terrible,[performance],negative,0.97
4,The staff were friendly,[staff],positive,0.90
5,the service was very slow,[service],negative,0.94
6,The price is high,[price],neutral,0.60
7,the quality is excellent,[quality],positive,0.96
8,The performance was bad at first,[performance],negative,0.95
9,the service was great,[service],positive,0.98


In [34]:
df_w_aspects = pd.DataFrame(r_w_aspects)
df_w_aspects.head(5)

,Phrase,Label,Score
0,I really loved it,positive,0.98
1,This was a complete waste of time,negative,0.98
2,I still regret buying it,negative,0.89
3,I hated the experience at first,negative,0.97
4,I liked it at first,positive,0.91


## 6. Conclusion

The updated approach improves the handling of comments  
that contain multiple aspects.

The model now provides more structured outputs by associating  
aspects with sentiment predictions.

However, there are still limitations:

- Processing time increases when handling multiple inputs

This limitation will be addressed in the next stage of the project.